# Recitation 5 — Hash Tables & Relational Algebra (Python)

Save this notebook in your `cs301-notebooks` project and open it in your IDE.
Use the project .venv Python / Python (CS301) kernel.
See [Recitation 0](https://harryc919.github.io/TA-CS301/recitation0.html) for setup.

Write your reasoning in the Markdown answer cells (double-click to edit, Shift+Enter to render).
Relational algebra subscripts are written in brackets here: σ[dept = 'CS'](Course) is σ with the subscript dept = 'CS'.
For Exercise 2, edit and run the starter definition, then run its self-check.
An untouched starter fails the check; that is expected. Keep the interface unchanged.
The Red-Black Tree section at the end is optional extra content.
Reference answers are not in this notebook; they are on the
[Recitation 5 page](https://harryc919.github.io/TA-CS301/recitation5.html). Nothing is submitted or graded.


## What we are practicing

This Tuesday session covers the hash-table lectures from the second half of Week 4 and the first Week 5 lectures on databases: the relational model and relational algebra. The Red-Black Tree appears only in the optional Extra section at the end.

- Explain what a hash function must do, compare the division and multiplication methods, and say what problem universal hashing solves.
- Trace insertions under linear probing and double hashing, and explain primary clustering using probe counts.
- Explain why deleting from an open-addressing table must leave a DELETED marker instead of an empty slot, and what that costs later.
- Implement an open-addressing map with double hashing using only plain arrays, and state its expected and worst-case costs.
- Compute the result of a relational algebra expression on a small instance, and write expressions for English queries.
- Diagnose wrong queries, and decide which algebraic rewrites are valid, with a counterexample when one is not.

Prerequisites: arrays and modular arithmetic, expected cost from earlier weeks, and sets and tuples. No SQL and no database software are used in this session.

Spend about 12 minutes on the two review sections, then about 45 minutes on the core practice: Exercise 1 (15 minutes) and Exercise 2 (30 minutes: 7 for the trace, 23 for the code). Leave about 10 minutes for discussion. Timing is a guide, not a completion requirement. The optional practice and the Red-Black Tree section at the end are for after the session or for students who finish early.

These exercises are optional and ungraded. Use hints when needed. AI tools are allowed during recitation, but you should be able to explain, verify, and modify your answer. Follow the instructor's separate assignment and exam policies.

## 1. Hash functions and open addressing

A hash table stores n keys in an array of m slots. A *hash function* h maps each key to a slot, and the goal is to spread keys evenly so that each operation looks at only a few slots. The *load factor* is α = n / m.

### Hash functions

- **Division:** h(k) = k mod m. Choose m prime and not close to a power of 2. With m = $2^{p}$, only the lowest p bits of the key matter, so keys that differ only in their high bits all collide.
- **Multiplication:** h(k) = ⌊m · (kA mod 1)⌋ for a constant 0 < A < 1, where kA mod 1 is the fractional part of kA. The choice of m matters much less, so a power of 2 is fine; A ≈ (√5 − 1) / 2 is a common choice.
- **Universal hashing:** pick the hash function *at random*, when the table is created, from a family in which any two distinct keys collide with probability at most 1 / m. For example, $h_{a,b}$(k) = ((ak + b) mod p) mod m, with p a prime larger than every key, a chosen from 1 … p − 1, and b from 0 … p − 1. No fixed input is bad for most choices, so the expected cost holds for every input, even one chosen by an adversary.

### Open addressing

In open addressing every key lives in the table itself. To insert k, try the slots h(k, 0), h(k, 1), h(k, 2), … (the *probe sequence*) until one is free. A search follows the same sequence and stops when it finds k or reaches an **empty** slot: if k had been inserted, it would have been placed there or earlier.

| Probing | h(k, i) | Strength | Weakness |
| --- | --- | --- | --- |
| Linear | (h′(k) + i) mod m | Simple; neighboring slots are cache-friendly | Primary clustering: occupied slots form long runs, and any key landing in a run extends it |
| Quadratic | (h′(k) + $c_{1}$i + $c_{2}i^{2}$) mod m | Breaks up long runs | Keys with the same start share a sequence (secondary clustering); for bad $c_{1}$, $c_{2}$, m it misses some slots |
| Double hashing | ($h_{1}$(k) + i · $h_{2}$(k)) mod m | Different keys get different step sizes; close to uniform hashing | Two hash computations; $h_{2}$(k) must never be 0 and must share no factor with m |

With m prime and 1 ≤ $h_{2}$(k) ≤ m − 1, $h_{2}$(k) and m share no factor, so the first m probes of double hashing visit every slot exactly once.

**Delete.** Emptying a slot would cut the probe sequences that passed through it, so a later search would stop early and miss keys stored further along. Instead, delete marks the slot DELETED (a *tombstone*): search treats it as occupied and keeps going, and insert may reuse it.

**Cost.** Under the uniform hashing assumption, an unsuccessful search makes at most 1 / (1 − α) probes in expectation, and a successful search at most (1 / α) ln(1 / (1 − α)). At α = 0.5 these are 2 and about 1.4; at α = 0.9 they are 10 and about 2.6. The worst case is Θ(m). Tombstones are not empty, so they lengthen searches as if they were keys; a table with many deletions is eventually rebuilt.

**Perfect hashing.** When the set of keys is fixed in advance, two levels of universal hashing give O(1) worst-case search with O(n) expected space: the first level splits the keys into buckets, and each bucket gets its own collision-free second-level table whose size is the square of the bucket's size.

## 2. The relational model and relational algebra

A **relation** is a set of tuples that all have the same attributes, its *schema*. A *key* is a set of attributes whose values identify a tuple. Relations are sets: there are no duplicate tuples and no order. Relational algebra is a small set of operators that each take relations and return a new relation, so expressions can be nested like arithmetic.

| Operator | Notation | Result | Attributes of the result |
| --- | --- | --- | --- |
| Selection | σ[c](R) | Tuples of R that satisfy condition c | Same as R |
| Projection | π[A](R) | Each tuple cut down to the attributes in A; duplicates removed | A |
| Rename | ρ[S](R) | R under a new name S (attributes may be renamed too) | Same values, new names |
| Union, difference | R ∪ S, R − S | Tuples in either; tuples in R but not in S | Same as R and S (they must match) |
| Cartesian product | R × S | Every pair of a tuple from R and a tuple from S | All of R's and all of S's |
| Theta-join (derived) | R ⋈[θ] S | σ[θ](R × S) | All of R's and all of S's |
| Natural join (derived) | R ⋈ S | Pairs that agree on *every* attribute name R and S share | Union of both, shared ones once |
| Intersection (derived) | R ∩ S | R − (R − S) | Same as R and S |

Two reading habits help. First, read an expression as a tree and evaluate it from the inside out. Second, English words such as "no", "never", or "only" usually call for a set difference: first find the tuples that *do* have the property, then subtract them. A condition such as dept ≠ 'CS' inside σ looks at one tuple at a time and cannot express "none of them".

> Typing notation: in plain text you can write σ[dept = 'CS'](Course) as `sigma[dept = 'CS'](Course)`, and use `pi[...]`, `rho[...]`, `join`, `union`, and `minus` the same way.

## Core practice · about 45 minutes

Try each prompt before checking the reference answer on the course page. Written answers can be short lists, small tables, or sentences. Run the code cells in this notebook (or your IDE); the self-checks tell you whether the implementation is correct.

1. Exercise 1 · relational algebra: read, write, diagnose — about 15 minutes.
2. Exercise 2 · build an open-addressing hash table — about 30 minutes: a 7-minute trace, then the code.

## Exercise 1 · Relational algebra: read, write, diagnose

**Four short parts · about 15 minutes.** A small course database has three relations. Keys are `sid` for Student, `cid` for Course, and (`sid`, `cid`) for Takes.

**Student**

| sid | sname | major |
| --- | --- | --- |
| 1 | Ana | CS |
| 2 | Ben | Math |
| 3 | Chen | CS |
| 4 | Dana | Physics |
| 5 | Eli | Math |

**Course**

| cid | title | dept |
| --- | --- | --- |
| C1 | Algorithms | CS |
| C2 | Databases | CS |
| M1 | Linear Algebra | Math |
| P1 | Mechanics | Physics |

**Takes**

| sid | cid | grade |
| --- | --- | --- |
| 1 | C1 | A |
| 1 | C2 | B |
| 2 | C1 | B |
| 2 | M1 | A |
| 3 | M1 | B |
| 3 | P1 | A |
| 4 | P1 | B |

1. **Read.** How many tuples and how many attributes do Student × Takes and Student ⋈ Takes have? Then compute π[sname](Student ⋈ Takes ⋈ σ[dept = 'CS'](Course)), and name the attributes each natural join matches on. Chen is a CS major: does Chen appear, and why?
2. **Write.** Write an expression for "names of students who have taken *no* CS course", and give its result on this instance.
3. **Diagnose.** A classmate answers part 2 with π[sname](Student ⋈ Takes ⋈ σ[dept ≠ 'CS'](Course)). Name one student it wrongly includes and one it wrongly leaves out, and explain each.
4. **Valid rewrite?** R and S below are relations; A is a set of attributes. For each claim, say true or false. Give a short reason if true, and a counterexample with one or two tuples per relation if false.

- **4a.** σ[c](R ⋈ S) = σ[c](R) ⋈ S, when c uses only attributes of R.
- **4b.** π[A](R ∪ S) = π[A](R) ∪ π[A](S), when R and S have the same attributes.
- **4c.** π[A](R − S) = π[A](R) − π[A](S), when R and S have the same attributes.

<details><summary>Hint 1 · Find who did, then subtract</summary>

For part 2, first write the set of `sid`s of students who took at least one CS course. Every student's `sid` minus that set leaves exactly the students you want, including anyone who took no course at all. Join back to Student to get names.

</details>

<details><summary>Hint 2 · Make two tuples collide</summary>

For 4c, use attributes (A, B). Choose one tuple for R and one for S that have the same A value but different B values. What is R − S, and what are π[A](R) and π[A](S)?

</details>

### Your answer — Exercise 1

1. **Read:** Tuples and attributes of Student × Takes and of Student ⋈ Takes; the result of the expression; the attributes each join matches on; whether Chen appears, and why.
2. **Write:** Your expression, and its result on this instance.
3. **Diagnose:** One student wrongly included, one wrongly left out, and why.
4. **Valid rewrite?**
   - 4a.
   - 4b.
   - 4c.


## Exercise 2 · Build an open-addressing hash table

**A trace, then the code · about 30 minutes.** Both parts use double hashing with $h_{1}$(k) = k mod m and $h_{2}$(k) = 1 + (k mod (m − 1)), so probe i looks at slot ($h_{1}$(k) + i · $h_{2}$(k)) mod m.

### Part A · Trace (about 7 minutes)

1. Take m = 11 and insert the keys 22, 1, 13, 11, 24, 33 in this order. Do it twice: once with **linear probing**, ($h_{1}$(k) + i) mod 11, and once with **double hashing**. For each, write the final slot of every key and the number of probes each insertion needed.
2. Compare the total number of probes. Where did linear probing lose, and why?
3. In the double-hashing table, delete 11 by simply making its slot *empty*. Then search for 33. What happens, and what should delete have done instead?

### Part B · Implement (about 23 minutes)

Implement a fixed-capacity map from non-negative integer keys to non-negative integer values, using open addressing with the double hashing above. **Use only plain arrays:** no dictionaries, sets, or maps from the language library (for example, Python `dict` or `set`, Java `HashMap`, C# `Dictionary`, C++ `unordered_map`, or JavaScript `Map`).

- `HashTable(m)` creates an empty table with m slots; m is a prime with 3 ≤ m ≤ 200,003.
- `put(key, value)` inserts the key, or updates its value if it is already stored, and returns true. It returns false only when the key is new and no EMPTY or DELETED slot is left.
- `get(key)` returns the stored value, or −1 if the key is absent.
- `remove(key)` marks the key's slot DELETED and returns true, or returns false if the key is absent.
- `size()` returns the number of keys stored (DELETED slots do not count).
- 0 ≤ key, value ≤ 1,000,000,000. The starter already has the three slot arrays, the slot states EMPTY, OCCUPIED, and DELETED, `size`, and `probe(key, i)`; you write `put`, `get`, and `remove`. No operation may probe more than m slots.

Two bugs are easy to write. The self-checks test both, and the written explanation asks about them.

1. **Bug 1 · Delete writes EMPTY.** With m = 7, put 10 (slot 3), then put 17: it collides at slot 3 and goes to slot 2. Remove 10. If slot 3 becomes EMPTY, `get(17)` stops at slot 3 and wrongly returns −1.
2. **Bug 2 · Put stops at the first DELETED slot.** Continue the example: slot 3 is now DELETED and 17 is still in slot 2. If `put(17, 3)` stores 17 in slot 3 right away, the table now holds 17 twice. Put should remember the first DELETED slot but keep probing until an EMPTY slot (or m probes) to make sure the key is not already stored further along.

### Examples and checks

| Operations (m = 7 unless stated) | Expected | What to check |
| --- | --- | --- |
| get(3) on a new table | −1 | Empty table |
| put(10, 100), put(10, 111), get(10), size() | true, true, 111, 1 | Update changes the value, not the size |
| put(10, 1), put(17, 2), remove(10), get(17) | …, true, 2 | Bug 1 |
| …then put(17, 3), size() | true, 1 | Bug 2: no duplicate key |
| m = 5: put five keys, then put a new one | false | Full table; updating a stored key still returns true |
| m = 5: put five keys, remove all five, get(10) | −1 | Only DELETED slots: the search must stop after m probes |

### Function interface (Python)

```python
class HashTable:
    def __init__(self, m: int)
    def put(self, key: int, value: int) -> bool
    def get(self, key: int) -> int
    def remove(self, key: int) -> bool
    def size(self) -> int
```

**Explain your answer:** in your own words, why must remove leave a DELETED marker, and why must put keep probing past the first DELETED slot? Give the expected and the worst-case number of probes per operation, and the space used. Why does double hashing need m to be prime here (or, more generally, $h_{2}$(k) to share no factor with m)? What happens to performance after many insertions and removals, even if size() stays small?

<details><summary>Hint · Compute h₁ and h₂ first</summary>

Write $h_{1}$ and $h_{2}$ for every key before you start: for example, 11 has $h_{1}$ = 0 and $h_{2}$ = 1 + (11 mod 10) = 2, so double hashing tries slots 0, 2, 4, 6, …

</details>

<details><summary>Hint 1 · Write get first</summary>

Loop i from 0 to m − 1 and compute slot = probe(key, i). If the slot is EMPTY, return −1. If it is OCCUPIED with this key, return its value. Otherwise (DELETED, or a different key) keep going. If the loop ends, return −1. Remove is the same loop, except that it marks the slot DELETED, decreases the count, and returns true.

</details>

<details><summary>Hint 2 · Put needs two pieces of memory</summary>

Keep `firstDeleted` (the first DELETED slot seen, or −1) and stop at the first EMPTY slot. If the loop finds the key, update it and return. Otherwise store the key in `firstDeleted` if there is one, else in the EMPTY slot where the loop stopped; if there is neither, the table is full.

</details>

Edit and run the starter definition in the next cell, then run its self-check.

In [ ]:
EMPTY, OCCUPIED, DELETED = 0, 1, 2


class HashTable:
    def __init__(self, m: int):
        # m is a prime >= 3, so every probe sequence visits all m slots.
        self.m = m
        self.count = 0  # number of OCCUPIED slots
        self.keys = [0] * m
        self.values = [0] * m
        self.state = [EMPTY] * m

    def probe(self, key: int, i: int) -> int:
        # Double hashing: h1(k) = k mod m, h2(k) = 1 + (k mod (m - 1)).
        return (key % self.m + i * (1 + key % (self.m - 1))) % self.m

    def put(self, key: int, value: int) -> bool:
        # TODO: for i = 0 .. m - 1, look at slot probe(key, i).
        # - OCCUPIED with this key: update the value and return True.
        # - DELETED: remember the first one, but keep probing.
        # - EMPTY: stop probing.
        # Then store the new key in the first DELETED slot if there was one,
        # otherwise in the EMPTY slot; return False if neither exists.
        pass

    def get(self, key: int) -> int:
        # TODO: probe until the key is found (return its value) or an EMPTY
        # slot is reached; stop after m probes. Return -1 if absent.
        pass

    def remove(self, key: int) -> bool:
        # TODO: find the key as in get. Mark its slot DELETED (not EMPTY),
        # update count, and return True. Return False if absent.
        pass

    def size(self) -> int:
        return self.count


In [ ]:
# Run this cell after implementing and running the HashTable cell.
# These checks fail while the starter is still unchanged.
def expect(actual, expected, label):
    assert actual == expected, f"{label}: expected {expected!r}, got {actual!r}"

t = HashTable(7)
expect(t.get(3), -1, "get on an empty table")
expect(t.remove(3), False, "remove on an empty table")
expect(t.put(10, 100), True, "insert 10")
expect(t.get(10), 100, "get after insert")
expect(t.put(10, 111), True, "update 10")
expect((t.get(10), t.size()), (111, 1), "update changes the value, not the size")
expect(t.put(0, 0), True, "key 0 with value 0")
expect(t.get(0), 0, "value 0 is a real value, not 'absent'")

# Bug 1: 10 -> slot 3; 17 collides at slot 3 and goes to slot 2.
t = HashTable(7)
t.put(10, 1)
t.put(17, 2)
expect(t.remove(10), True, "remove 10")
expect(t.get(17), 2, "bug 1: after removing 10, 17 must still be found (use DELETED, not EMPTY)")
expect(t.remove(10), False, "removing 10 twice")
# Bug 2: slot 3 is now DELETED; 17 is still stored further along in slot 2.
expect(t.put(17, 3), True, "update 17")
expect(t.size(), 1, "bug 2: put must not stop at the first DELETED slot and store a duplicate 17")
expect(t.remove(17), True, "remove 17")
expect(t.get(17), -1, "bug 2: no stale copy of 17 is left behind")

# Part A table: remove 11, then 33 (probes 0 -> 4 -> 8) must still be found.
t = HashTable(11)
for k in [22, 1, 13, 11, 24, 33]:
    t.put(k, k * 10)
t.remove(11)
expect(t.get(33), 330, "Part A: 33 after removing 11")

# Full table, then a table with only DELETED slots.
t = HashTable(5)
for k in range(5):
    expect(t.put(k * 5, k), True, f"fill slot {k}")
expect(t.put(99, 1), False, "full table: a new key is rejected")
expect(t.put(10, 7), True, "full table: updating an existing key still works")
for k in range(5):
    t.remove(k * 5)
expect(t.get(10), -1, "all slots DELETED: search must stop after m probes")
expect(t.put(99, 1), True, "all slots DELETED: insert reuses one")
expect((t.get(99), t.size()), (1, 1), "after reusing a DELETED slot")

# Random operations against a direct-address table (keys are small here).
import random
rng = random.Random(301)
for m in [3, 7, 101]:
    t, oracle, size = HashTable(m), [-1] * (3 * m), 0
    for step in range(2000):
        key, op = rng.randrange(3 * m), rng.random()
        if op < 0.5:
            value, ok = rng.randrange(1000), oracle[key] != -1 or size < m
            expect(t.put(key, value), ok, f"m = {m}, step {step}: put({key}, {value})")
            if ok:
                size += oracle[key] == -1
                oracle[key] = value
        elif op < 0.75:
            expect(t.remove(key), oracle[key] != -1, f"m = {m}, step {step}: remove({key})")
            size -= oracle[key] != -1
            oracle[key] = -1
        else:
            expect(t.get(key), oracle[key], f"m = {m}, step {step}: get({key})")
        expect(t.size(), size, f"m = {m}, step {step}: size()")

# Larger table: 60,000 keys in 100,003 slots (load factor about 0.6).
t = HashTable(100_003)
keys = [i * 7919 for i in range(60_000)]
for k in keys:
    t.put(k, k % 1_000_000)
for k in keys[::2]:
    t.remove(k)
assert t.size() == 30_000, "large table: size after 60,000 puts and 30,000 removes"
assert all(t.get(k) == (-1 if i % 2 == 0 else k % 1_000_000) for i, k in enumerate(keys)), "large table: wrong get"
print("Exercise 2 checks passed (bugs 1 and 2, full and all-DELETED tables, 6,000 random operations, 100,003 slots).")


### Your answer — Exercise 2

**Part A · Trace**

| Key | h₁ | h₂ | Linear: slots tried | Double: slots tried |
|---|---|---|---|---|
| 22 | | | | |
| 1 | | | | |
| 13 | | | | |
| 11 | | | | |
| 24 | | | | |
| 33 | | | | |

- **Total probes and clustering:**
- **Searching for 33 after emptying 11's slot:**

**Part B · Explain your code**

- **DELETED, not EMPTY:**
- **Put keeps probing past the first DELETED slot:**
- **Cost:** Expected and worst-case probes per operation, and space.
- **Why m prime:**
- **After many insertions and removals:**


## Optional practice

Short written questions for after the session, or for students who finish early. They are not on the core path.

1. **Division with a power of 2.** With h(k) = k mod 16, where do the keys 16, 32, 48, 64 go? Where do they go with m = 17, and why does a prime help?
2. **Quadratic probing that gets stuck.** With h(k, i) = (h′(k) + $i^{2}$) mod 8 and h′(k) = 0, list the slots tried for i = 0, 1, …, 7. Can an insertion fail even though the table still has empty slots?
3. **Why random?** Explain why choosing the hash function at random from a universal family protects against an adversary who picks the keys, when no single fixed hash function can.
4. **Self-join.** Using Exercise 1's instance, write an expression for "pairs of students (by `sid`) who have taken at least one course together". List each pair once, and never pair a student with themself.
5. **"Every" without division.** Write "students (by `sid`) who have taken *every* CS course" using only σ, π, ×, and −, and give its result on Exercise 1's instance. (If your lecture covered the division operator ÷, compare with π[sid, cid](Takes) ÷ π[cid](σ[dept = 'CS'](Course)).)

<details><summary>Hint · Rename, and build what is missing</summary>

For question 4, two copies of Takes are needed: rename them, for example T1 and T2, and join them on the course with a condition that orders the two `sid`s. For question 5, build every (student, CS course) pair with ×, subtract the pairs that exist in Takes, and see which students are left with a missing pair.

</details>

### Your answer — Optional practice (optional)

1. **Division with m = 16 and m = 17:**
2. **Quadratic probing slots:**
3. **Why random:**
4. **Self-join:**
5. **Every CS course:**


## Discussion and exit check

In Part A, linear and double hashing put the same six keys into the same table. Which one needed fewer probes, and would the difference grow or shrink as the table fills?

Compare bug 1 and bug 2 from Exercise 2. Each comes from reasoning about a single slot when the correct answer depends on the whole probe sequence. Where else in this session did a decision about one item need information about a whole set?

**Exit check:** a hash table and the query "students who took no CS course" both have to handle something that is *absent*. How does each one decide that something is absent?

Before leaving, keep your own copy of the notebook or code, and write down one remaining question for the TA.

### Your reflection

**Linear vs double hashing:** Which needed fewer probes, and how does the gap change as the table fills?

**One slot vs the whole sequence:** Where else did a decision about one item need information about a whole set?

**Exit check:** How do a hash table and the "no CS course" query each decide that something is absent?


## Extra · Red-Black Trees

This section is provided by the TA for students who want to go further. It is not covered in lectures or on the final exam, and the core exercises do not depend on it.

### Why balance a search tree?

A hash table gives expected O(1) operations but keeps no order: it cannot efficiently report the smallest key, the next key after k, or all keys in a range. A binary search tree (BST) can do all of these, but each operation costs O(h) for a tree of height h, and inserting keys in sorted order makes h = n. A **red-black tree** is a BST that stays balanced, with h ≤ 2 $\log_{2}$(n + 1), so search, insert, and delete all take O(log n) time in the worst case.

### The five properties

1. Every node is red or black.
2. The root is black.
3. Every leaf (the empty NIL child below a node) is black.
4. A red node has two black children: no two red nodes are adjacent on a path.
5. For each node, every path from it down to a NIL leaf contains the same number of black nodes, its *black-height*.

**Why the height is at most 2 $\log_{2}$(n + 1).** By induction, a subtree whose root has black-height b contains at least $2^{b}$ − 1 keys. By property 4, at least half the nodes on any root-to-leaf path are black, so the root's black-height is at least h / 2. Therefore n ≥ $2^{h/2}$ − 1, which gives h ≤ 2 $\log_{2}$(n + 1).

### Rotations

A *left rotation* at node x, whose right child is y, makes y the root of that subtree and x its left child; y's old left subtree becomes x's right subtree. A right rotation is the mirror image. A rotation changes O(1) pointers and keeps the BST order, because every key keeps the same position in an in-order traversal.

### Insertion

Insert the new key z as in an ordinary BST and color it **red**. This keeps property 5, but property 4 breaks if z's parent is also red. While that happens, look at z's *uncle* (the parent's sibling). The table describes the case where z's parent is a left child; the other case is the mirror image.

| Case | Situation | Fix |
| --- | --- | --- |
| 1 | Uncle is red | Color the parent and uncle black and the grandparent red, then continue from the grandparent (two levels up) |
| 2 | Uncle is black, and z is a right child (a "triangle") | Left-rotate at the parent, and let z be the old parent; this turns the triangle into case 3 |
| 3 | Uncle is black, and z is a left child (a "line") | Right-rotate at the grandparent, then swap the colors of the old parent and grandparent; done |

At the end, color the root black. Case 1 may repeat up the tree, but cases 2 and 3 happen at most once, so insertion makes at most two rotations and O(log n) recolorings. Deletion also takes O(log n) time with at most three rotations; it has more cases and is not covered here.

### Worked example

Insert 10, 20, 30, 15, 25, 27 into an empty tree. Trees are written as key and color, then the left and right subtrees in parentheses; · is an empty child.

| Insert | What happens | Tree afterwards |
| --- | --- | --- |
| 10 | New root, colored black | 10B |
| 20 | Red child of a black node: nothing to fix | 10B(·, 20R) |
| 30 | Parent 20 is red, uncle is NIL (black), 30 is in a line with 20 and 10 (mirror of case 3): left-rotate at 10, swap colors | 20B(10R, 30R) |
| 15 | Parent 10 is red, uncle 30 is red (case 1): recolor 10 and 30 black, 20 red; the root turns black again | 20B(10B(·, 15R), 30B) |
| 25 | Parent 30 is black: nothing to fix | 20B(10B(·, 15R), 30B(25R, ·)) |
| 27 | Parent 25 is red, uncle is NIL, 27 is a right child of a left child (case 2): left-rotate at 25; then case 3: right-rotate at 30, color 27 black and 30 red | 20B(10B(·, 15R), 27B(25R, 30R)) |

**Try it:** insert 13 into the final tree 20B(10B(·, 15R), 27B(25R, 30R)). Which cases apply, and what is the tree afterwards? Check that every path from the root to a NIL leaf has the same number of black nodes.

### Your answer — Red-Black Tree (optional)

**Inserting 13:** The cases that apply, the tree afterwards, and your check of the black-heights.


## References and related material

- T. H. Cormen, C. E. Leiserson, R. L. Rivest, and C. Stein, *Introduction to Algorithms*, 4th ed. (MIT Press, 2022): Chapter 11 (Hash Tables) and Chapter 13 (Red-Black Trees).
- A. Silberschatz, H. F. Korth, and S. Sudarshan, *Database System Concepts*, 7th ed. (McGraw-Hill, 2019): Chapter 2 (Introduction to the Relational Model), including relational algebra.
- [Recitation 2 · growth bounds](https://harryc919.github.io/TA-CS301/recitation2.html) — for reading the expected and worst-case costs.
- [Recitation 0 · notebook environment and kernels](https://harryc919.github.io/TA-CS301/recitation0.html)